In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd

In [6]:
# recipes = pd.read_csv('/content/drive/MyDrive/recipes_data_10k.csv')
# grams_matrix = pd.read_csv('/content/drive/MyDrive/recipes_ingredients_grams_matrix.csv')
# unique_cost = pd.read_csv('/content/drive/MyDrive/unique_ingredients_with_cost.csv')
# ingredients_nutr = pd.read_csv('/content/drive/MyDrive/ingredients_nutrition_content.csv')
import os
print(os.getcwd())
recipes = pd.read_csv('../recipes_data_10k.csv')
grams_matrix = pd.read_csv('../compute-cost/recipes_ingredients_grams_matrix.csv')
unique_cost = pd.read_csv('../compute-cost/unique_ingredients_with_cost.csv')
ingredients_nutr = pd.read_csv('../ingredients_nutrition_content.csv')


/Users/joshirajaram/Documents/MS-CS/ECS289A/ECS289A-CostAwareRecipeGenerator/src


In [7]:
grams_long = grams_matrix.melt(
    id_vars=['title'],
    var_name='ingredient',
    value_name='grams'
)

# keep only ingredients that appear (grams > 0)
grams_long = grams_long[grams_long['grams'] > 0].reset_index(drop=True)

In [8]:
grams_long.shape

(68149, 3)

In [9]:
recipes = recipes.reset_index().rename(columns={'index': 'recipe_id'})
# join recipe_id onto grams_long by title
grams_long = grams_long.merge(recipes[['recipe_id', 'title']],
                              on='title',
                              how='left')

In [10]:
grams_long.shape

(246790, 4)

In [11]:
grams_long[grams_long['title'] == "Acini De Pepe"]

,title,ingredient,grams,recipe_id
1,Acini De Pepe,acini,100,5233
83254,Acini De Pepe,eggs,100,5233
93249,Acini De Pepe,flour,180,5233
119465,Acini De Pepe,mandarin oranges,45,5233
172607,Acini De Pepe,pineapple,2,5233
189664,Acini De Pepe,salt,25,5233
213781,Acini De Pepe,sugar,454,5233


In [ ]:
!pip install networkx node2vec

import networkx as nx
from node2vec import Node2Vec

G = nx.Graph()

# Add nodes
for rid in grams_long['recipe_id'].unique():
    G.add_node(f"r:{rid}", bipartite='recipe')

for ing in grams_long['ingredient'].unique():
    G.add_node(f"i:{ing}", bipartite='ingredient')

# Add edges with weight=grams
for _, row in grams_long.iterrows():
    rid = row['recipe_id']
    ing = row['ingredient']
    grams_val = float(row['grams'])
    G.add_edge(f"r:{rid}", f"i:{ing}", weight=grams_val)

In [ ]:
node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=20,
    num_walks=30,
    workers=4
)

w2v_model = node2vec.fit(window=10, min_count=1, batch_words=4)

Computing transition probabilities:   0%|          | 0/13630 [00:00<?, ?it/s]

In [ ]:
w2v_model.save('/content/drive/MyDrive/node2vec_ingredient_embeddings.model')

In [ ]:
import pickle
with open('/content/drive/MyDrive/node2vec_graph.pkl', 'wb') as f:
    pickle.dump(G, f)

In [1]:
!pip install networkx node2vec
!pip install gensim

In [15]:
pip install numpy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
from gensim.models import Word2Vec
import pickle

w2v_model = Word2Vec.load('../node2vec_ingredient_embeddings.model')

with open('../node2vec_graph.pkl', 'rb') as f:
    G = pickle.load(f)

In [17]:
import numpy as np

ingredient_nodes = [n for n in G.nodes() if n.startswith('i:')]
rows = []

for node in ingredient_nodes:
    ing_name = node[2:]  # strip off "i:"
    vec = w2v_model.wv[node]
    rows.append([ing_name] + list(vec))

emb_dim = len(rows[0]) - 1
emb_cols = [f'emb_{i}' for i in range(emb_dim)]

ing_emb = pd.DataFrame(rows, columns=['ingredient'] + emb_cols)

In [19]:
ing_full = (unique_cost
            .merge(ing_emb, on='ingredient', how='inner'))

In [20]:
ing_full['price_per_g'] = ing_full['approx_cost_per_100g_usd'] / 100.0

In [21]:
ing_full.shape

(3703, 67)

In [22]:
!pip install rapidfuzz

# from rapidfuzz import process, fuzz

# ingredients_nutr['desc_lower'] = ingredients_nutr['Description'].str.lower()
# ing_full['ing_lower'] = ing_full['ingredient'].str.lower()

# # build a lookup list
# descs = ingredients_nutr['desc_lower'].tolist()

# matches = []
# for ing in ing_full['ing_lower']:
#     match, score, idx = process.extractOne(
#         ing, descs, scorer=fuzz.token_set_ratio
#     )
#     matches.append((ing, match, score, idx))

# matches_df = pd.DataFrame(matches, columns=['ing_lower','matched_desc','score','idx'])

# # keep only reasonably similar matches (e.g. score >= 80)
# matches_df = matches_df[matches_df['score'] >= 80]

# # attach Nutrient Data Bank Number back to ing_full
# ingredients_nutr = ingredients_nutr.reset_index().rename(columns={'index': 'nutr_idx'})
# nutr_map = matches_df.merge(ingredients_nutr[['nutr_idx','Description'] + [c for c in ingredients_nutr.columns if c.startswith('Data.')]],
#                             left_on='idx',
#                             right_on='nutr_idx',
#                             how='left')

# # merge nutr_map into ing_full via ing_lower
# ing_full = ing_full.merge(nutr_map.drop(columns=['idx','nutr_idx']),
#                           on='ing_lower', how='left')


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 1.9 MB/s eta 0:00:000:00:010:00:01:01

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [23]:
from rapidfuzz import process, fuzz

# Lowercase for fuzzy matching
ingredients_nutr['desc_lower'] = ingredients_nutr['Description'].str.lower()
ing_full['ing_lower'] = ing_full['ingredient'].str.lower()

desc_list = ingredients_nutr['desc_lower'].tolist()

matches = []
for ing in ing_full['ing_lower'].unique():
    match, score, idx = process.extractOne(
        ing,
        desc_list,
        scorer=fuzz.token_set_ratio
    )
    matches.append((ing, match, score, idx))

matches_df = pd.DataFrame(matches, columns=['ing_lower','matched_desc','score','idx'])

# keep only reasonably confident matches
matches_df = matches_df[matches_df['score'] >= 80]

# attach nutrition rows
# The error "ValueError: The column label 'nutr_idx' is not unique." indicates that
# 'ingredients_nutr' has multiple columns named 'nutr_idx'. This typically happens
# if 'ingredients_nutr.reset_index().rename(columns={'index': 'nutr_idx'})' is executed
# multiple times without proper cleanup.
# To fix this, we first ensure unique column names, keeping the first 'nutr_idx' if duplicates exist.
ingredients_nutr = ingredients_nutr.loc[:, ~ingredients_nutr.columns.duplicated(keep='first')].copy()
ingredients_nutr = ingredients_nutr.reset_index().rename(columns={'index': 'nutr_idx'})

nutr_cols_raw = [          # calories per 100g
    'Data.Protein',           # protein per 100g
    'Data.Fat.Total Lipid',   # fat per 100g
    'Data.Carbohydrate'       # carbs per 100g
]

matches_with_nutr = matches_df.merge(
    ingredients_nutr[['nutr_idx','desc_lower'] + nutr_cols_raw],
    left_on='idx',
    right_on='nutr_idx',
    how='left'
)

# merge into ing_full by ing_lower
ing_full = ing_full.merge(
    matches_with_nutr[['ing_lower'] + nutr_cols_raw],
    on='ing_lower', how='left'
)

# convert to per-gram
for c in nutr_cols_raw:
    ing_full[c + '_per_g'] = ing_full[c] / 100.0

In [24]:
ing_full.head(50)

,ingredient,approx_cost_per_100g_usd,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,...,emb_62,emb_63,price_per_g,ing_lower,Data.Protein,Data.Fat.Total Lipid,Data.Carbohydrate,Data.Protein_per_g,Data.Fat.Total Lipid_per_g,Data.Carbohydrate_per_g
0,achiote coloring,1.0,-0.264100,-0.205410,0.040562,-0.353823,0.463045,-0.485216,-0.169213,-0.186562,...,0.395713,-0.404864,0.010,achiote coloring,NaN,NaN,NaN,NaN,NaN,NaN
1,acini,1.0,0.275882,-0.167725,0.483716,-0.360426,0.091280,-0.993323,0.172372,0.647950,...,-0.484683,-0.321431,0.010,acini,NaN,NaN,NaN,NaN,NaN,NaN
2,acorn squash,1.0,-0.021688,0.341616,0.313980,0.146956,0.060776,-0.609525,0.118476,-1.003707,...,0.193751,-0.146417,0.010,acorn squash,NaN,NaN,NaN,NaN,NaN,NaN
3,active dry yeast,1.0,0.358005,-0.444868,0.785679,-0.202455,0.003067,-0.551830,-0.214757,0.488112,...,0.407465,-0.374615,0.010,active dry yeast,NaN,NaN,NaN,NaN,NaN,NaN
4,active yeast,1.0,0.226836,-0.933807,0.356888,0.227303,-0.312603,-0.463950,0.380832,0.563667,...,0.311728,-0.377881,0.010,active yeast,NaN,NaN,NaN,NaN,NaN,NaN
5,adams wheat beer,1.0,0.086800,-0.167590,0.662300,0.354208,0.253713,-0.293951,-0.065937,-0.147307,...,0.220790,-0.903438,0.010,adams wheat beer,NaN,NaN,NaN,NaN,NaN,NaN
6,ajinomoto,1.0,0.383090,0.144515,-0.072783,-0.053381,-0.130146,0.254305,0.115053,-0.018861,...,-0.239867,-0.612283,0.010,ajinomoto,NaN,NaN,NaN,NaN,NaN,NaN
7,alaga syrup,1.0,0.621936,-0.291341,-0.121285,0.413650,0.292986,-0.590800,-0.099686,-0.068078,...,-0.100437,-0.040986,0.010,alaga syrup,NaN,NaN,NaN,NaN,NaN,NaN
8,alfalfa honey,1.0,0.083121,-0.286617,0.237290,0.004613,0.287616,-0.004451,0.487887,0.302045,...,0.125645,-0.425671,0.010,alfalfa honey,0.30,0.00,82.40,0.0030,0.0000,0.8240
9,alfalfa sprouts,1.0,0.249209,-0.042977,1.600509,-0.697986,-1.161154,-0.479925,-0.649614,0.156132,...,-1.243407,-0.572494,0.010,alfalfa sprouts,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
ing_full.shape

(3703, 74)

In [26]:
print(ing_full[nutr_cols_raw].notna().mean())

Data.Protein            0.178504
Data.Fat.Total Lipid    0.178504
Data.Carbohydrate       0.178504
dtype: float64


In [27]:
nutr_cols_raw = [
    'Data.Protein',          # grams per 100g
    'Data.Fat.Total Lipid',
    'Data.Carbohydrate'
]

for c in nutr_cols_raw:
    ing_full[c + '_per_g'] = ing_full[c] / 100.0

In [28]:
print(ingredients_nutr.columns.tolist())

['nutr_idx', 'Category', 'Description', 'Nutrient Data Bank Number', 'Data.Alpha Carotene', 'Data.Beta Carotene', 'Data.Beta Cryptoxanthin', 'Data.Carbohydrate', 'Data.Cholesterol', 'Data.Choline', 'Data.Fiber', 'Data.Lutein and Zeaxanthin', 'Data.Lycopene', 'Data.Niacin', 'Data.Protein', 'Data.Retinol', 'Data.Riboflavin', 'Data.Selenium', 'Data.Sugar Total', 'Data.Thiamin', 'Data.Water', 'Data.Fat.Monosaturated Fat', 'Data.Fat.Polysaturated Fat', 'Data.Fat.Saturated Fat', 'Data.Fat.Total Lipid', 'Data.Major Minerals.Calcium', 'Data.Major Minerals.Copper', 'Data.Major Minerals.Iron', 'Data.Major Minerals.Magnesium', 'Data.Major Minerals.Phosphorus', 'Data.Major Minerals.Potassium', 'Data.Major Minerals.Sodium', 'Data.Major Minerals.Zinc', 'Data.Vitamins.Vitamin A - RAE', 'Data.Vitamins.Vitamin B12', 'Data.Vitamins.Vitamin B6', 'Data.Vitamins.Vitamin C', 'Data.Vitamins.Vitamin E', 'Data.Vitamins.Vitamin K', 'desc_lower']


In [29]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Embedding columns
emb_cols = [c for c in ing_full.columns if c.startswith('emb_')]

# Choose nutrition features
nutr_cols = [
    'Data.Protein_per_g',
    'Data.Fat.Total Lipid_per_g',
    'Data.Carbohydrate_per_g'
]

def macro_vec(row):
    return row[nutr_cols].values.astype(float)

def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)

def get_ing_row(ingredient_name):
    row = ing_full[ing_full['ingredient'] == ingredient_name]
    if row.empty:
        return None
    return row.iloc[0]

def suggest_substitute_for_ing(ingredient_name, grams_orig,
                               alpha=0.5, beta=0.3, gamma=0.2,
                               top_k=5):
    row_i = get_ing_row(ingredient_name)
    if row_i is None:
        raise ValueError(f"Unknown ingredient: {ingredient_name}")

    emb_i = row_i[emb_cols].values.astype(float)
    m_i = macro_vec(row_i)
    T = grams_orig * m_i
    price_i = row_i['price_per_g']

    candidates = []

    for _, row_j in ing_full.iterrows():
        j_name = row_j['ingredient']
        if j_name == ingredient_name:
            continue

        m_j = macro_vec(row_j)
        if np.linalg.norm(m_j) < 1e-9:
            continue

        # least squares grams for substitute
        g_sub = np.dot(T, m_j) / (np.dot(m_j, m_j) + 1e-9)
        if g_sub <= 0 or g_sub > 3 * grams_orig:  # heuristic cap
            continue

        nutr_j = g_sub * m_j
        nutr_diff = np.linalg.norm(nutr_j - T) / (np.linalg.norm(T) + 1e-9)

        cost_orig = grams_orig * price_i
        cost_new = g_sub * row_j['price_per_g']
        if cost_new >= cost_orig:
            continue  # not cheaper

        cost_saving = cost_orig - cost_new
        norm_cost_saving = cost_saving / (cost_orig + 1e-9)

        emb_j = row_j[emb_cols].values.astype(float)
        sim_embed = cosine(emb_i, emb_j)

        score = alpha * sim_embed - beta * nutr_diff + gamma * norm_cost_saving

        candidates.append({
            'ingredient_sub': j_name,
            'grams_sub': g_sub,
            'sim_embed': sim_embed,
            'nutr_diff': nutr_diff,
            'cost_orig': cost_orig,
            'cost_new': cost_new,
            'cost_saving': cost_saving,
            'score': score
        })

    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)
    return candidates[:top_k]


In [30]:
import numpy as np

emb_cols = [c for c in ing_full.columns if c.startswith('emb_')]
nutr_cols = [
    'Data.Protein_per_g',
    'Data.Fat.Total Lipid_per_g',
    'Data.Carbohydrate_per_g'
]

def macro_vec(row):
    v = row[nutr_cols].values.astype(float)
    if np.isnan(v).any():
        return None
    return v

def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)

def get_ing_row(name):
    r = ing_full[ing_full['ingredient'] == name]
    return r.iloc[0] if not r.empty else None

def suggest_substitute_for_ing(ingredient_name, grams_orig,
                               alpha=0.5, beta=0.3, gamma=0.2,
                               top_k=5):
    row_i = get_ing_row(ingredient_name)
    if row_i is None:
        raise ValueError(f"Unknown ingredient: {ingredient_name}")

    if np.isnan(row_i['price_per_g']):
        raise ValueError(f"No price for ingredient: {ingredient_name}")

    m_i = macro_vec(row_i)
    if m_i is None:
        raise ValueError(f"No nutrition for ingredient: {ingredient_name}")

    emb_i = row_i[emb_cols].values.astype(float)
    T = grams_orig * m_i
    price_i = row_i['price_per_g']

    candidates = []

    for _, row_j in ing_full.iterrows():
        j_name = row_j['ingredient']
        if j_name == ingredient_name:
            continue

        if np.isnan(row_j['price_per_g']):
            continue

        m_j = macro_vec(row_j)
        if m_j is None or np.linalg.norm(m_j) < 1e-9:
            continue

        # least-squares grams for substitute
        g_sub = np.dot(T, m_j) / (np.dot(m_j, m_j) + 1e-9)
        if g_sub <= 0 or g_sub > 3 * grams_orig:
            continue

        nutr_j = g_sub * m_j
        nutr_diff = np.linalg.norm(nutr_j - T) / (np.linalg.norm(T) + 1e-9)

        cost_orig = grams_orig * price_i
        cost_new = g_sub * row_j['price_per_g']
        if cost_new >= cost_orig:
            continue

        cost_saving = cost_orig - cost_new
        norm_cost_saving = cost_saving / (cost_orig + 1e-9)

        emb_j = row_j[emb_cols].values.astype(float)
        sim_embed = cosine(emb_i, emb_j)

        score = alpha * sim_embed - beta * nutr_diff + gamma * norm_cost_saving

        candidates.append({
            'ingredient_sub': j_name,
            'grams_sub': g_sub,
            'sim_embed': sim_embed,
            'nutr_diff': nutr_diff,
            'cost_orig': cost_orig,
            'cost_new': cost_new,
            'cost_saving': cost_saving,
            'score': score
        })

    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)
    return candidates[:top_k]


In [31]:
r = grams_long[grams_long['title'] == "Creamy Corn"]

for _, row in r.iterrows():
    ing = row['ingredient']
    grams_orig = row['grams']
    print(f"\nOriginal ingredient: {ing} ({grams_orig} g)")
    try:
        subs = suggest_substitute_for_ing(ing, grams_orig)
        for s in subs:
            print(f"  -> {s['ingredient_sub']} ({s['grams_sub']:.1f} g), "
                  f"cost {s['cost_new']:.3f} vs {s['cost_orig']:.3f}, "
                  f"nutr_diff={s['nutr_diff']:.3f}, score={s['score']:.3f}")
    except ValueError as e:
        print("  (no data)", e)




Original ingredient: butter (2 g)
  -> puffed wheat (0.1 g), cost 0.001 vs 0.024, nutr_diff=1.000, score=0.311
  -> salt (2.5 g), cost 0.005 vs 0.024, nutr_diff=0.003, score=0.282
  -> frozen white bread (0.1 g), cost 0.001 vs 0.024, nutr_diff=1.000, score=0.254
  -> unsalted butter (2.5 g), cost 0.005 vs 0.024, nutr_diff=0.008, score=0.250
  -> golden grahams cereal (0.1 g), cost 0.001 vs 0.024, nutr_diff=0.999, score=0.247

Original ingredient: cream cheese (80 g)
  -> pecan nuts (31.8 g), cost 0.318 vs 0.960, nutr_diff=0.135, score=0.315
  -> garlic cheese roll (21.3 g), cost 0.256 vs 0.960, nutr_diff=0.903, score=0.314
  -> pack cream cheese (80.0 g), cost 0.960 vs 0.960, nutr_diff=0.000, score=0.304
  -> pecans (31.8 g), cost 0.318 vs 0.960, nutr_diff=0.135, score=0.272
  -> coconut (23.1 g), cost 0.231 vs 0.960, nutr_diff=0.263, score=0.263

Original ingredient: frozen corn (907 g)
  (no data) No nutrition for ingredient: frozen corn

Original ingredient: garlic powder (2 g)
  -

In [32]:
nutr_cols_raw = [c for c in ing_full.columns if c.startswith("Data.")]
ing_full[nutr_cols_raw].isna().mean().head()  # this will be ~1.0 for most columns

Data.Protein                  0.821496
Data.Fat.Total Lipid          0.821496
Data.Carbohydrate             0.821496
Data.Protein_per_g            0.821496
Data.Fat.Total Lipid_per_g    0.821496
dtype: float64

In [33]:
print(ing_full.shape)
print(ing_full['ingredient'].head())

(3703, 74)
0    achiote coloring
1               acini
2        acorn squash
3    active dry yeast
4        active yeast
Name: ingredient, dtype: object


In [34]:
valid_price_mask = (
    ing_full['price_per_g'].notna() &
    (ing_full['price_per_g'] > 0)
)

print("With non-null & non-zero price:", valid_price_mask.sum())


print("Total ingredients in ing_full:", len(ing_full))
print("With non-null price:", ing_full['price_per_g'].notna().sum())
print("With full nutrition:", ing_full[nutr_cols].notna().all(axis=1).sum())

With non-null & non-zero price: 3703
Total ingredients in ing_full: 3703
With non-null price: 3703
With full nutrition: 661


In [35]:
valid_price_mask = (
    ing_full['price_per_g'].notna() &
    (ing_full['price_per_g'] > 0)
)

print("With non-null & non-zero price:", valid_price_mask.sum())


print("Total ingredients in ing_full:", len(ing_full))
print("With non-null price:", ing_full['price_per_g'].notna().sum())
print("With full nutrition:", ing_full[nutr_cols].notna().all(axis=1).sum())

With non-null & non-zero price: 3703
Total ingredients in ing_full: 3703
With non-null price: 3703
With full nutrition: 661


In [39]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.2/261.2 kB 10.6 MB/s eta 0:00:00


In [44]:
# In the left pane, store gemini API key in secrets
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [46]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI learns from data to make decisions or predictions.


In [36]:
grams_long[grams_long['title'] == 'Acini De Pepe']

,title,ingredient,grams,recipe_id
1,Acini De Pepe,acini,100,5233
83254,Acini De Pepe,eggs,100,5233
93249,Acini De Pepe,flour,180,5233
119465,Acini De Pepe,mandarin oranges,45,5233
172607,Acini De Pepe,pineapple,2,5233
189664,Acini De Pepe,salt,25,5233
213781,Acini De Pepe,sugar,454,5233


In [37]:
def get_ings_by_cost_decreasing_order(title, top_k=3):
  recipe = grams_long[grams_long['title'] == title]
  ings = recipe['ingredient']
  cost = []
  for i in ings:
    p = ing_full[ing_full['ing_lower'] == i.lower()]
    cost.append({
        'ingredient': i,
        'cost': recipe[recipe['ingredient'] == i]['grams'].iloc[0] * p['price_per_g'].iloc[0],
        'grams': recipe[recipe['ingredient'] == i]['grams'].iloc[0]
    })
  cost.sort(key=lambda x:x['cost'], reverse=True)
  return cost[:top_k]

In [38]:
! pip install ollama

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [42]:
import ollama
import json

# PASTE YOUR PUBLIC NGROK URL HERE
NGROK_URL = "http://127.0.0.1:11434" # Corrected URL format
MODEL_NAME = "llama3.2:3b" # Use the model name you tested locally

client = ollama.Client(host=NGROK_URL)

prompt_text = "Write a short, non-rhyming poem about the silence of code."

# Use the client.generate method
try:
    response = client.generate(
        model=MODEL_NAME,
        prompt=prompt_text,
        options={
            "temperature": 0.5, # Optional: makes the output less random
            "seed": 42         # Optional: for reproducible results
        }
        # The API call defaults to stream=False if not specified here
    )

    # The result is a dictionary. The generated text is in the 'response' key.
    print("\n--- Model Response (Non-Streaming) ---")
    print(response['response'])
    print("--------------------------------------")
except Exception as e:
    print(f"ERROR: Could not connect to the remote Ollama host at {NGROK_URL}")
    print("Ensure the ngrok tunnel is still active and the URL is correct.")
    print(e)


--- Model Response (Non-Streaming) ---
In the depths of a computer's mind
A world of ones and zeros resides
A language devoid of sound or motion
Where meaning is forged from empty space

The silence of code is a heavy blanket
That shrouds the soul of innovation
A quiet discipline that demands precision
And a patient hand that weaves its logic

In this realm, beauty is not for show
But rather an intricate balance of form and function
A symphony of 1s and 0s
Where every note is a calculated step towards perfection
--------------------------------------


In [53]:
import time
original_ing = []
substituted_ing = []
original_cost = []
new_cost = []
original_recipes = []
generated_recipes = []
recipe_titles = []
count = 0
start_time = time.time()
print(f"Script started at: {time.ctime(start_time)}")
for i in range(1, 1001):
  title = recipes.iloc[i]['title']
  ingredients = recipes.iloc[i]['NER']
  directions = recipes.iloc[i]['directions']
  r = get_ings_by_cost_decreasing_order(title, top_k=1)
  # print('---------------------------------------------')
  # print(title, r)
  for row in r:
    ing = row['ingredient']
    grams_orig = row['grams']
    # print(f"\nOriginal ingredient: {ing} ({grams_orig} g)")
    try:
        subs = suggest_substitute_for_ing(ing, grams_orig, top_k=3)
        for s in subs:
            # print(f"  -> {s['ingredient_sub']} ({s['grams_sub']:.1f} g), "
            #       f"cost {s['cost_new']:.3f} vs {s['cost_orig']:.3f}, "
            #       f"nutr_diff={s['nutr_diff']:.3f}, score={s['score']:.3f}"
            #       f"sim_embed={s['sim_embed']:.3f}"
            # )
            system_prompt = f"""You are a pro chef and you are asked to create a cheaper meal alternative.
            This is the original recipe title: {title}.
            These are the ingredients: {ingredients}.
            These are the directions: {directions}.
            Now generate recipe directions where you substitute {ing} with {s['ingredient_sub']} in this recipe.
            Only give the directions of the new recipe, don't give anything else. Give the output in a single line. 
            Be very brief and concise. This is important."""
            # print(system_prompt)
            # time.sleep(1)
            response = client.generate(
                model=MODEL_NAME,
                prompt=system_prompt,
                options={
                    # "temperature": 0.5, # Optional: makes the output less random
                    "seed": 42         # Optional: for reproducible results
                }
                # The API call defaults to stream=False if not specified here
            )
            # response = client.models.generate_content(
            #     model="gemini-2.0-flash-lite",
            #     contents=system_prompt,
            # )
            count += 1
            # print("Original Recipe", directions)
            # print("Generated Recipe", response.text)
            original_recipes.append(''.join(directions))
            generated_recipes.append(response['response'])
            original_ing.append(ing)
            substituted_ing.append(s['ingredient_sub'])
            original_cost.append(s['cost_orig'])
            new_cost.append(s['cost_new'])
            recipe_titles.append(title)
    except ValueError as e:
        print("  (no data)", e)
    if i % 20 == 0:
      print(response['response'])
    # print(i, "done")
print(len(generated_recipes), count)
data = {
    'Title': recipe_titles,
    'OriginalRecipe': original_recipes,
    'OriginalIngredient': original_ing,
    'SubstitutedIngredient': substituted_ing,
    'GeneratedRecipe': generated_recipes,
    'OriginalCost': original_cost,
    'NewCost': new_cost
}

# 2. Convert the dictionary to a DataFrame
df = pd.DataFrame(data)

# 3. Save the DataFrame to a CSV file
df.to_csv('generated_recipes_1_1000.csv', index=False)
end_time = time.time()
print(f"Script ended at: {time.ctime(end_time)}")

# --- Calculate Duration ---
duration = end_time - start_time

print(f"\nTotal execution duration: {duration:.4f} seconds")

Script started at: Mon Nov 24 04:45:47 2025
  (no data) No nutrition for ingredient: frozen corn
  (no data) No nutrition for ingredient: shredded cheese
  (no data) No nutrition for ingredient: extra lean ground beef
  (no data) No nutrition for ingredient: flaked coconut
  (no data) No nutrition for ingredient: condensed milk
  (no data) No nutrition for ingredient: nutmeg
"Replace shortening with 1:1 ratio of unsalted butter; cream sugar and butter instead."
  (no data) No nutrition for ingredient: oregano
  (no data) No nutrition for ingredient: beef stock
  (no data) No nutrition for ingredient: condensed milk
  (no data) No nutrition for ingredient: fruit cocktail
  (no data) No nutrition for ingredient: oregano
  (no data) No nutrition for ingredient: ground nuts
  (no data) No nutrition for ingredient: pimentos
  (no data) No nutrition for ingredient: vegetable soup mix
  (no data) No nutrition for ingredient: salad supreme
  (no data) No nutrition for ingredient: lemon yogurt
